<a href="https://www.kaggle.com/code/abhishekgodara/cafa-6-protein-prediction?scriptVersionId=292371156" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
import os
import gc
import pandas as pd
import numpy as np
import networkx as nx
from tqdm.auto import tqdm
from collections import defaultdict

# ==========================================
# CONFIGURATION
# ==========================================
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
SUBMISSION_INPUT = '/kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv'
SUBMISSION_OUTPUT = 'submission.tsv'

# Adjustable parameters for score improvement
MIN_SCORE_THRESHOLD = 0.001  # Remove noise
SCORE_ROUNDING = 3  # Decimal places
USE_SCORE_BOOST = True  # Enable intelligent boosting
USE_PERCENTILE_SCALING = True  # Scale based on percentile distribution
TOP_K_PREDICTIONS = None  # Set to limit predictions per protein (None = no limit)

# ==========================================
# 1. IMPROVED GRAPH BUILDER WITH ANCESTOR CACHE
# ==========================================
def build_go_graph_with_cache(go_obo_path):
    """Build GO graph with cached ancestors for faster propagation"""
    print(f"[1/5] Parsing OBO and building ancestor cache...")
    go_graph = nx.DiGraph()
    
    if not os.path.exists(go_obo_path):
        raise FileNotFoundError(f"OBO not found: {go_obo_path}")

    # Parse OBO
    with open(go_obo_path, "r") as f:
        cur_id = None
        for line in f:
            line = line.strip()
            if line == "[Term]":
                cur_id = None
            elif line.startswith("id: "):
                cur_id = line.split("id: ")[1].strip()
                go_graph.add_node(cur_id)
            elif line.startswith("is_a: "):
                pid = line.split()[1].strip()
                if cur_id:
                    go_graph.add_edge(cur_id, pid)
            elif line.startswith("relationship: part_of "):
                parts = line.split()
                if len(parts) >= 3:
                    pid = parts[2].strip()
                    if cur_id:
                        go_graph.add_edge(cur_id, pid)

    print("      -> Validating DAG and computing ancestors...")
    
    # Remove cycles if any
    try:
        cycles = list(nx.simple_cycles(go_graph))
        if cycles:
            print(f"      -> Removing {len(cycles)} cycles...")
            for cycle in cycles:
                if len(cycle) > 1:
                    go_graph.remove_edge(cycle[0], cycle[1])
    except:
        pass
    
    # Build ancestor cache (much faster than repeated traversals)
    print("      -> Building ancestor cache...")
    ancestors_cache = {}
    topo_order = list(nx.topological_sort(go_graph))
    
    # Process in reverse topological order (roots to leaves)
    for node in reversed(topo_order):
        all_ancestors = set()
        for successor in go_graph.successors(node):
            all_ancestors.add(successor)
            all_ancestors.update(ancestors_cache.get(successor, set()))
        ancestors_cache[node] = all_ancestors
    
    print(f"      -> Graph built: {len(go_graph.nodes())} terms, {len(go_graph.edges())} edges")
    
    return go_graph, ancestors_cache, topo_order

# ==========================================
# 2. INTELLIGENT SCORE OPTIMIZATION
# ==========================================
def optimize_scores_for_protein(protein_scores, ancestors_cache, protein_id=""):
    """
    Optimize scores for a single protein with multiple strategies
    """
    if not protein_scores:
        return protein_scores
    
    # Convert to dict for faster operations
    scores_dict = protein_scores.copy()
    
    # STRATEGY 1: HARD MAX PROPAGATION
    propagated_scores = scores_dict.copy()
    
    # Propagate scores upward using ancestor cache
    for term, score in sorted(scores_dict.items(), key=lambda x: x[1], reverse=True):
        if score <= 0:
            continue
            
        # Get all ancestors from cache
        for ancestor in ancestors_cache.get(term, set()):
            current = propagated_scores.get(ancestor, 0.0)
            if score > current:
                propagated_scores[ancestor] = score
    
    # STRATEGY 2: SELECTIVE SCORE BOOSTING
    if USE_SCORE_BOOST and propagated_scores:
        # Identify top predictions to boost
        scores_list = list(propagated_scores.items())
        scores_list.sort(key=lambda x: x[1], reverse=True)
        
        # Only boost if we have meaningful predictions
        if len(scores_list) >= 3:
            top_score = scores_list[0][1]
            second_score = scores_list[1][1]
            
            # Boost strategy based on score distribution
            if top_score < 0.8:
                # If top score is low, boost it moderately
                boost_factor = min(1.5, 0.85 / max(top_score, 0.01))
                
                # Apply boost with diminishing returns
                for i, (term, score) in enumerate(scores_list):
                    if i < 5:  # Boost top 5 predictions
                        # Stronger boost for higher ranks
                        rank_boost = 1.0 + (0.3 * (1.0 - i/5))
                        new_score = min(1.0, score * boost_factor * rank_boost)
                        propagated_scores[term] = new_score
                    elif score < 0.1 and i > 10:
                        # Suppress very low scores that aren't top predictions
                        propagated_scores[term] = score * 0.7
    
    # STRATEGY 3: PERCENTILE-BASED SCALING
    if USE_PERCENTILE_SCALING and propagated_scores:
        scores_array = np.array(list(propagated_scores.values()))
        
        if len(scores_array) > 10:
            # Calculate percentiles
            p90 = np.percentile(scores_array, 90)
            p50 = np.percentile(scores_array, 50)
            
            if p90 > 0 and p50 > 0:
                # Scale to make distribution more favorable for CAFA
                # CAFA's F-max benefits from confident predictions
                scale_factor = min(2.0, 0.7 / max(p50, 0.01))
                
                # Apply scaling with upper bound
                for term in list(propagated_scores.keys()):
                    new_score = min(1.0, propagated_scores[term] * scale_factor)
                    propagated_scores[term] = new_score
    
    # STRATEGY 4: ENSURE ROOT TERMS
    # In GO, these are the main roots (optional, but can help consistency)
    roots = {'GO:0003674', 'GO:0008150', 'GO:0005575'}
    for root in roots:
        if root in ancestors_cache or any(root in ancestors_cache.get(t, set()) for t in propagated_scores):
            # If any prediction exists in this branch, ensure root is predicted
            max_score = max([propagated_scores.get(t, 0) for t in propagated_scores 
                           if root in ancestors_cache.get(t, set())] + [0])
            if max_score > 0.1:
                propagated_scores[root] = max(propagated_scores.get(root, 0), 0.95)
    
    # Filter low scores
    filtered_scores = {k: v for k, v in propagated_scores.items() 
                      if v >= MIN_SCORE_THRESHOLD}
    
    return filtered_scores

# ==========================================
# 3. CHUNKED PROCESSING FOR MEMORY EFFICIENCY
# ==========================================
def process_submission_chunked(submission_df, ancestors_cache, chunk_size=50000):
    """Process large submissions in chunks to save memory"""
    print(f"[3/5] Processing predictions in chunks...")
    
    # Get unique proteins
    unique_proteins = submission_df['protein_id'].unique()
    num_chunks = (len(unique_proteins) + chunk_size - 1) // chunk_size
    
    all_results = []
    
    for chunk_idx in tqdm(range(num_chunks), desc="Processing chunks"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(unique_proteins))
        chunk_proteins = unique_proteins[start_idx:end_idx]
        
        # Filter dataframe for this chunk
        chunk_df = submission_df[submission_df['protein_id'].isin(chunk_proteins)]
        
        # Process each protein in chunk
        grouped = chunk_df.groupby('protein_id')
        
        for pid, group in grouped:
            # Convert to dict
            protein_scores = dict(zip(group['go_term'], group['score']))
            
            # Optimize scores
            optimized_scores = optimize_scores_for_protein(protein_scores, ancestors_cache, pid)
            
            # Collect results
            for term, score in optimized_scores.items():
                all_results.append((pid, term, float(score)))
        
        # Clear memory
        if chunk_idx % 10 == 0:
            gc.collect()
    
    return pd.DataFrame(all_results, columns=['protein_id', 'go_term', 'score'])

# ==========================================
# 4. POST-PROCESSING OPTIMIZATIONS
# ==========================================
def post_process_predictions(df):
    """Apply final optimizations to improve CAFA score"""
    print(f"[4/5] Post-processing predictions...")
    
    # 1. Remove exact duplicates
    df = df.drop_duplicates(subset=['protein_id', 'go_term'])
    
    # 2. Sort by protein and score
    df = df.sort_values(['protein_id', 'score'], ascending=[True, False])
    
    # 3. Apply top-k filtering if enabled
    if TOP_K_PREDICTIONS:
        print(f"      -> Limiting to top {TOP_K_PREDICTIONS} predictions per protein")
        df = df.groupby('protein_id').head(TOP_K_PREDICTIONS).reset_index(drop=True)
    
    # 4. Round scores to reduce file size and help thresholding
    df['score'] = df['score'].round(SCORE_ROUNDING)
    
    # 5. Final threshold (after rounding)
    df = df[df['score'] >= MIN_SCORE_THRESHOLD]
    
    # 6. Ensure no scores exceed 1.0
    df['score'] = df['score'].clip(0.0, 1.0)
    
    return df

# ==========================================
# 5. MAIN PIPELINE
# ==========================================
def main():
    print("=" * 70)
    print("CAFA 6 SUBMISSION OPTIMIZER")
    print(f"Target: Improve from baseline 0.378")
    print("=" * 70)
    
    # 1. Build GO graph with ancestor cache
    go_graph, ancestors_cache, topo_order = build_go_graph_with_cache(OBO_PATH)
    
    # 2. Load submission
    print(f"\n[2/5] Loading submission from {SUBMISSION_INPUT}...")
    
    # Check file size to decide on chunking
    file_size = os.path.getsize(SUBMISSION_INPUT)
    print(f"      -> File size: {file_size / 1024 / 1024:.1f} MB")
    
    # Load with appropriate settings
    submission = pd.read_csv(
        SUBMISSION_INPUT, 
        sep='\t', 
        header=None,
        names=['protein_id', 'go_term', 'score'],
        dtype={'protein_id': str, 'go_term': str, 'score': np.float32},
        on_bad_lines='skip'
    )
    
    print(f"      -> Loaded {len(submission):,} predictions")
    print(f"      -> Unique proteins: {submission['protein_id'].nunique():,}")
    
    # Basic stats
    print(f"\n      Score Statistics:")
    print(f"        Min: {submission['score'].min():.4f}")
    print(f"        Max: {submission['score'].max():.4f}")
    print(f"        Mean: {submission['score'].mean():.4f}")
    print(f"        Median: {submission['score'].median():.4f}")
    
    # 3. Process predictions
    if len(submission) > 1000000:
        # Use chunked processing for large files
        print(f"\n      -> Large file detected, using chunked processing...")
        optimized_df = process_submission_chunked(submission, ancestors_cache)
    else:
        # Process all at once
        print(f"\n      -> Processing all predictions at once...")
        all_results = []
        grouped = submission.groupby('protein_id')
        
        for pid, group in tqdm(grouped, total=len(grouped), desc="Processing proteins"):
            protein_scores = dict(zip(group['go_term'], group['score']))
            optimized_scores = optimize_scores_for_protein(protein_scores, ancestors_cache, pid)
            
            for term, score in optimized_scores.items():
                all_results.append((pid, term, float(score)))
        
        optimized_df = pd.DataFrame(all_results, columns=['protein_id', 'go_term', 'score'])
    
    # 4. Post-processing
    final_df = post_process_predictions(optimized_df)
    
    # 5. Save
    print(f"\n[5/5] Saving optimized submission...")
    final_df.to_csv(SUBMISSION_OUTPUT, sep='\t', index=False, header=False)
    
    # 6. Final statistics
    print("\n" + "=" * 70)
    print("OPTIMIZATION COMPLETE")
    print("=" * 70)
    print(f"\nResults Summary:")
    print(f"  • Input predictions: {len(submission):,}")
    print(f"  • Output predictions: {len(final_df):,}")
    print(f"  • Unique proteins: {final_df['protein_id'].nunique():,}")
    print(f"  • Unique GO terms: {final_df['go_term'].nunique():,}")
    
    print(f"\nScore Distribution (optimized):")
    for threshold in [0.01, 0.1, 0.3, 0.5, 0.7, 0.9]:
        count = (final_df['score'] >= threshold).sum()
        percentage = (count / len(final_df)) * 100
        print(f"  • ≥{threshold:.2f}: {count:,} ({percentage:.1f}%)")
    
    
    print(f"\nOutput saved to: {SUBMISSION_OUTPUT}")

if __name__ == "__main__":
    main()

CAFA 6 SUBMISSION OPTIMIZER
Target: Improve from baseline 0.378
[1/5] Parsing OBO and building ancestor cache...
      -> Validating DAG and computing ancestors...
      -> Building ancestor cache...
      -> Graph built: 48106 terms, 69001 edges

[2/5] Loading submission from /kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv...
      -> File size: 1430.8 MB
      -> Loaded 40,796,692 predictions
      -> Unique proteins: 279,437

      Score Statistics:
        Min: 0.0027
        Max: 48.1258
        Mean: 0.2405
        Median: 0.0896

      -> Large file detected, using chunked processing...
[3/5] Processing predictions in chunks...


Processing chunks:   0%|          | 0/6 [00:00<?, ?it/s]

[4/5] Post-processing predictions...

[5/5] Saving optimized submission...

OPTIMIZATION COMPLETE

Results Summary:
  • Input predictions: 40,796,692
  • Output predictions: 55,558,156
  • Unique proteins: 279,437
  • Unique GO terms: 33,896

Score Distribution (optimized):
  • ≥0.01: 55,558,153 (100.0%)
  • ≥0.10: 35,321,613 (63.6%)
  • ≥0.30: 23,830,660 (42.9%)
  • ≥0.50: 19,567,842 (35.2%)
  • ≥0.70: 17,960,178 (32.3%)
  • ≥0.90: 12,080,914 (21.7%)

Expected Improvements:
  • Hard max propagation: Ensures ontological consistency
  • Selective boosting: Strengthens confident predictions
  • Percentile scaling: Improves score distribution for F-max
  • Root term handling: Adds missing high-level predictions

Output saved to: submission.tsv

Next steps: Try adjusting parameters:
  1. Set TOP_K_PREDICTIONS = 1500 to reduce noise
  2. Adjust MIN_SCORE_THRESHOLD (0.001-0.01)
  3. Tune boost factors in optimize_scores_for_protein()
